# Cultura Database — Getting Started

This notebook shows how to load and query the Cultura Database using Python.

**Example**: Extract all scientists from contemporary China (1900–2024) and plot their number over time.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

## 1. Connect to the Database

Download `cultura_database.db` from [OSF](https://osf.io/) and place it in the same directory as this notebook.

In [ ]:
conn = sqlite3.connect("cultura_database.db")

# Quick check: how many individuals in the database?
total = pd.read_sql_query("SELECT COUNT(*) as n FROM individuals", conn)
print(f"Total individuals: {total['n'][0]:,}")

## 2. Extract Chinese Scientists (1900–2024)

We filter on:
- `nationalities` containing a Chinese nationality (e.g., "People's Republic of China")
- `occupations` containing "scientist" or any sub-occupation of scientist
- `birthdate` between 1900 and 2024

In [ ]:
query = """
SELECT wikidata_id, name, birthdate, occupations, nationalities
FROM individuals
WHERE (
    nationalities LIKE '%People\'s Republic of China%'
    OR nationalities LIKE '%Republic of China%'
    OR nationalities LIKE '%Chinese%'
)
AND birthdate IS NOT NULL
AND birthdate >= '1900'
AND birthdate < '2025'
"""

df = pd.read_sql_query(query, conn)
print(f"Chinese individuals born 1900-2024: {len(df):,}")
df.head(10)

In [ ]:
# Parse birth year
df["birth_year"] = pd.to_numeric(df["birthdate"].str[:4], errors="coerce")
df = df.dropna(subset=["birth_year"])
df["birth_year"] = df["birth_year"].astype(int)

# Filter to scientists (occupations containing science-related terms)
scientist_keywords = ["scientist", "physicist", "chemist", "biologist", "mathematician",
                      "engineer", "researcher", "astronomer", "geologist", "computer scientist"]

mask = df["occupations"].fillna("").apply(
    lambda x: any(kw in x.lower() for kw in scientist_keywords)
)
scientists = df[mask].copy()
print(f"Chinese scientists born 1900-2024: {len(scientists):,}")

## 3. Plot the Number of Scientists by Decade

In [ ]:
scientists["decade"] = (scientists["birth_year"] // 10) * 10
by_decade = scientists.groupby("decade").size().reset_index(name="count")

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(by_decade["decade"], by_decade["count"], width=8, color="#c0392b", edgecolor="white")
ax.set_xlabel("Birth Decade")
ax.set_ylabel("Number of Scientists")
ax.set_title("Scientists from China in the Cultura Database (born 1900–2024)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Explore the Data Further

You can easily adapt this notebook for other countries, time periods, or occupation types.

In [ ]:
# Example: top 10 occupations among Chinese individuals (1900-2024)
all_occupations = df["occupations"].dropna().str.split("; ").explode()
top_occ = all_occupations.value_counts().head(10)
print("Top 10 occupations (Chinese individuals, 1900-2024):")
print(top_occ.to_string())

In [ ]:
conn.close()